In [596]:
"""
This script creates artificial data for a discrete choice problem.
Assume there are three modes of transportation to choose from. Six fixed
variables were designed as significant and five as non-significant.
Seven random variables (five normal, one uniform, one triangular) were
designed as significant.
Three normal variables were correlated.
Two normal variables were non-linearly transformed.
"""


'\nThis script creates artificial data for a discrete choice problem.\nAssume there are three modes of transportation to choose from. Six fixed\nvariables were designed as significant and five as non-significant.\nSeven random variables (five normal, one uniform, one triangular) were\ndesigned as significant.\nThree normal variables were correlated.\nTwo normal variables were non-linearly transformed.\n'

In [597]:

import numpy as np

import random





import statsmodels.api as sm
from scipy.stats import multivariate_normal

In [598]:
import numpy as np
import pandas as pd
import scipy.stats as ss

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
is_correlated = 0

    
if is_correlated:
    name_2 = 'artificial_mixed_corr_2023_MOOF.csv'
else:
    name_2 = 'artificial_ZA.csv'
    
is_panel = 1
if is_panel:
    name_2 = 'panel_synth.csv'


In [599]:
def noise(n_obs, perc=1, random_state=None):
    random_state = random_state or np.random
    noise_vec = random_state.normal(0, 1, n_obs)
    return noise_vec

In [600]:
def random_col(N, P, J, random_state=None, low = 5, high = 25, noise_val  = 0.0):
    rand_nums = random_state.randint(low=low, high=high, size=(N,))/8
    return np.repeat(rand_nums, P) + noise_val*noise(N*P*J, random_state=random_state)

def generate_random_df(N, P, J, num_fixed=0, num_isvars=0, num_randvars=0, random_state=None, G = None):
    df = pd.DataFrame()
    #df['choice_id'] = np.repeat(np.arange(1, (N*P+1)), J)
    df['ind_id'] = np.repeat(np.arange(1, N+1), P*J)
    #df['alt'] = np.tile(np.arange(1, J+1), N*P)

    varnames = []
    
    
    for i in range(num_isvars):
        coef_name = 'added_isvar' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, low = 0, high = 10, noise_val = .5)
    
    
    for i in range(num_fixed):
        coef_name = 'added_fixed' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, low = 0, high = 9, noise_val = 0.1)

    
        

    for i in range(num_randvars):
        coef_name = 'added_random' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, low = 0, high =9, noise_val=.1)
        
    for i in range(1):
        coef_name = 'constant'
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, low = 1, high =2)*8 
    if G is not None:
        df['group'] = np.repeat(np.arange(1, N/G+1),P*J*G)

    return df, varnames


In [601]:

N =10000  # Number of observations
P = 1  # Number of choices per individual
J = 1  # Number of alternatives
G =200
num_fixed = 0
num_isvars = 3
num_nonsig = 3
num_randvars = 3
seed = 122
random_state = np.random.RandomState(seed)
np.random.seed(seed)

#random.seed(seed)
df, varnames = generate_random_df(N, P, J, num_fixed=num_fixed, num_isvars=num_isvars,
                                  num_randvars=num_randvars, random_state=random_state, G = G)

# Shift values <-2 for as inverse boxcox transform will be applied



np.shape(df)

(10000, 9)

In [602]:
df.head(40)

,ind_id,added_isvar1,added_isvar2,added_isvar3,added_random1,added_random2,added_random3,constant,group
0,1,-0.024227,1.004291,0.040551,1.029894,0.041551,0.404511,1.0,1.0
1,2,-0.196461,-0.385949,0.113323,0.202645,-0.078452,0.736039,1.0,1.0
2,3,0.203832,0.248302,-0.076271,0.708162,0.917509,0.638694,1.0,1.0
3,4,1.344469,0.255069,0.091709,0.215819,0.552910,1.019604,1.0,1.0
4,5,-0.841789,1.247249,0.827866,0.077555,0.985833,0.937711,1.0,1.0
5,6,1.118729,1.050897,1.621369,0.053406,0.468567,0.026441,1.0,1.0
6,7,0.252337,-0.029274,1.022421,0.728982,0.226912,0.863862,1.0,1.0
7,8,1.177454,0.888835,0.645216,0.402970,0.189340,0.499644,1.0,1.0
8,9,-0.171624,0.087127,0.191826,0.041239,0.950302,0.878980,1.0,1.0
9,10,0.069942,0.620585,1.692525,0.041778,0.450792,1.068934,1.0,1.0


In [603]:
# Define coefficients (betas)
# Fixed betas
fixed_coefs = [ random_state.choice([1,2,3 , 3, 2,3]) *random_state.uniform(1, 4) for i in range(num_fixed)]
fixed_coefs = np.array(fixed_coefs)



fixed_coefs = list(fixed_coefs)


In [604]:
fixed_coefs

[]

In [605]:
# Random mean between -1.5 and 1.5, excluding -.1 - .1 as hard to detect effect
random_coefs_mean = [random_state.choice([1,6, -2, -3,4, 2, 2, 3, -2]) * random_state.uniform(.5, 2.5) for i in range(num_randvars)]
random_coefs_sd = [random_state.uniform(.5, 2) for i in range(num_randvars)]
print(random_coefs_sd)
cov_mat = np.diag(random_coefs_sd)
cov_mat[0, 1] = cov_mat[1, 0] = 0
cov_mat[0, 2] = cov_mat[2, 0] = 0.0
cov_mat[1, 2] = cov_mat[2, 1] = 0

random_coefs_uniform_a = 0
random_coefs_uniform_b = random_state.uniform(2, 4)

random_coefs_tri_left = 0
random_coefs_tri_right = random_state.uniform(1, 2)
random_coefs_tri_mode = random_coefs_tri_right/2

rand_coefs = [np.array([]) for i in range(num_randvars)]

a = int(N/G)
draws = list()
ndraws = 500
for rr in range(ndraws):
    
    for i in range(a):
        res_normal =np.array(random_state.normal(random_coefs_mean[0], random_coefs_sd[0]))
        #res_normal = random_state.multivariate_normal(random_coefs_mean, cov_mat)
      #  res_uniform = np.array([random_state.uniform(random_coefs_uniform_a, random_coefs_uniform_b)])
        res_uniform = np.array(random_state.normal(random_coefs_mean[1], random_coefs_sd[1]))
      #  res_triangular = np.array([random_state.triangular(random_coefs_tri_left, random_coefs_tri_mode, random_coefs_tri_right)])
        res_triangular = np.array(random_state.normal(random_coefs_mean[2], random_coefs_sd[2]))
        
        res = np.hstack((res_normal, res_uniform, res_triangular))
        #res = res_normal
        for r in range(num_randvars):
        # print(res[r])
            
          rand_coefs[r] =  np.repeat(res[r], a*P*J*G)
            #print(rand_coefs[r])
    draws.append(rand_coefs) 
    #rand_coefs = [np.array([]) for i in range(num_randvars)]     
rand_coefs = draws

[1.9104328245157571, 1.2575562688649609, 1.566649829727359]


In [606]:
rand_coefs

[[array([-2.70678076, -2.70678076, -2.70678076, ..., -2.70678076,
         -2.70678076, -2.70678076]),
  array([4.08025242, 4.08025242, 4.08025242, ..., 4.08025242, 4.08025242,
         4.08025242]),
  array([8.91779615, 8.91779615, 8.91779615, ..., 8.91779615, 8.91779615,
         8.91779615])],
 [array([-2.70678076, -2.70678076, -2.70678076, ..., -2.70678076,
         -2.70678076, -2.70678076]),
  array([4.08025242, 4.08025242, 4.08025242, ..., 4.08025242, 4.08025242,
         4.08025242]),
  array([8.91779615, 8.91779615, 8.91779615, ..., 8.91779615, 8.91779615,
         8.91779615])],
 [array([-2.70678076, -2.70678076, -2.70678076, ..., -2.70678076,
         -2.70678076, -2.70678076]),
  array([4.08025242, 4.08025242, 4.08025242, ..., 4.08025242, 4.08025242,
         4.08025242]),
  array([8.91779615, 8.91779615, 8.91779615, ..., 8.91779615, 8.91779615,
         8.91779615])],
 [array([-2.70678076, -2.70678076, -2.70678076, ..., -2.70678076,
         -2.70678076, -2.70678076]),
  a

In [607]:
for r in range(num_randvars):
    print(r)
#res_uniform
#res_triangular
#random_coefs_mean
#random_coefs_sd

0
1
2


In [608]:
random_coefs_tri_left, random_coefs_tri_mode, random_coefs_tri_right

(0, 0.7310408716326854, 1.4620817432653708)

In [609]:
np.size(rand_coefs)
P*N*J
np.shape(rand_coefs)
rand_coefs = np.transpose(rand_coefs)

In [610]:
B_fixed = [np.repeat(f_coef, P*N*J) for f_coef in fixed_coefs]
B_const = [np.repeat(3, P*N*J)]
if is_correlated:
    B_const = [np.repeat(-3, P*N*J)]
else:
    B_const = [np.repeat(-7, P*N*J)]
    


# Convert betas to matrix for easy product


In [611]:
rand_coefs = np.array(rand_coefs)
rand_coefs.shape
B_const = np.array(B_const).transpose()
B_const.shape

(10000, 1)

In [612]:
# Multiply and generate probability
#isvars = ['added_isvar' + str(i+1) for i in range(num_isvars)]
Xf = df.values[:, num_fixed+num_randvars+num_isvars+1:num_fixed+num_randvars+2+num_isvars] 
Xr = df.values[:, 1+num_isvars:num_fixed+num_randvars+2+num_isvars-1]  # Extract only necessary columns
print(Xf.shape)
rand_coefs = np.array(rand_coefs)
print(np.shape(rand_coefs))
#XB =      (X*B).sum(axis = 1).reshape(N*P, J)
#result = np.einsum('ij,ij->ij', a, b)

#print(B_const.shape, 'plea')
#Vdf = np.einsum('nk,nk -> n,k', Xf, B_const) # (N, K)
Vdr = np.einsum("nk,nkr -> nr", Xr, rand_coefs)  # (N,P,R)
print(Vdr.shape, 
      'd')
Vdboth = B_const[:, :, None] + Vdr[:, None, :]
print(Vdboth.shape, 'f')
XB = np.exp(np.clip(Vdboth, None, 700)).sum(axis = 2)/ndraws
scale = 1
dispersion = 3

eps = np.random.gumbel(0, 1, (N*P, J))
print(max(XB))
#XB = np.clip(XB, None, 7)
eXB = (XB).ravel()
dispersion = 10
scale = 1



#xg = np.array(rgamma(N, dispersion, dispersion))
xbg = eXB
#xbg = eXB

print(max(eXB))
nbyo = np.random.poisson(xbg)

# Use monte carlo simulation to predict choice
# y = np.apply_along_axis(lambda p: np.eye(J, dtype='int64')[np.argmax(p)], 1, prob).reshape(N*P*J,)
# y = y.reshape(N*P*J,)
print('max is', max(nbyo))
y = []
U = XB + eps


df['Y'] = nbyo
count = len(list(filter(lambda x: x != 0, nbyo)))
print(count)  # Output: 4


# Apply non-linear transformations for boxcox testing
save = "C:/Users/n9471103/source/repos/MetaCount/" +name_2
# Save to CSV
print(df.head())
#df = df.drop(columns = ['id'])
print(df.head())
df.to_csv(save, index=False)

(10000, 1)
(10000, 3, 500)
(10000, 500) d
(10000, 1, 500) f
[4845.8533923]
4845.853392302986
max is 4844
3766
   ind_id  added_isvar1  added_isvar2  added_isvar3  added_random1  \
0       1     -0.024227      1.004291      0.040551       1.029894   
1       2     -0.196461     -0.385949      0.113323       0.202645   
2       3      0.203832      0.248302     -0.076271       0.708162   
3       4      1.344469      0.255069      0.091709       0.215819   
4       5     -0.841789      1.247249      0.827866       0.077555   

   added_random2  added_random3  constant  group    Y  
0       0.041551       0.404511       1.0    1.0    0  
1      -0.078452       0.736039       1.0    1.0    1  
2       0.917509       0.638694       1.0    1.0    3  
3       0.552910       1.019604       1.0    1.0   46  
4       0.985833       0.937711       1.0    1.0  189  
   ind_id  added_isvar1  added_isvar2  added_isvar3  added_random1  \
0       1     -0.024227      1.004291      0.040551       1.029

In [613]:
# np.linalg.norm(model.coeff_[:11] - fixed_coefs)